In [1]:
# One-time environment setup (run once, then restart the kernel)
%pip install -q "langchain-community>=0.4,<0.4.2" "tenacity>=8.2.3,<9.0.0" --force-reinstall
%pip install -q langchain-ollama
print("RESTART THE KERNEL.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchainhub 0.1.21 requires packaging<25,>=23.2, but you have packaging 26.3 which is incompatible.
langgraph-sdk 0.4.3 requires websockets<17,>=14, but you have websockets 17.1 which is incompatible.
google-genai 2.22.0 requires websockets<17.0,>=13.0.0, but you have websockets 17.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Now RESTART THE KERNEL before running anything below.


In [1]:
%load_ext autoreload
%autoreload 2

import sys, os, json
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")

In [2]:
# Vectorstore, LLM, embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini"),
)
#llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

ragas_llm = ChatOllama(
    model="qwen2.5:7b",
    base_url="http://<4050-laptop-ip>:11434",
    temperature=0,
)

In [3]:
# Strategies
from rag_lab.strategies.naive import NaiveStrategy
from rag_lab.strategies.multi_query import MultiQueryStrategy
from rag_lab.strategies.rag_fusion import RagFusionStrategy
from rag_lab.strategies.hyde import HydeStrategy
from rag_lab.strategies.step_back import StepBackStrategy
from rag_lab.strategies.decomposition import DecompositionStrategy

strategies = {
    "naive": NaiveStrategy(vectorstore, llm=llm),
    "multi_query": MultiQueryStrategy(vectorstore, llm=llm),
    "rag_fusion": RagFusionStrategy(vectorstore, llm=llm),
    "hyde": HydeStrategy(vectorstore, llm=llm),
    "step_back": StepBackStrategy(vectorstore, llm=llm),
    "decomposition": DecompositionStrategy(vectorstore, llm=llm),
}

In [4]:
# Eval set
with open(project_root / "eval" / "eval_set.json") as f:
    eval_set = json.load(f)["questions"]
print(f"{len(eval_set)} questions loaded")

63 questions loaded


In [9]:
# Generation loop
from rag_lab.utils import call_with_backoff, is_daily_quota_error, load_completed_keys, append_jsonl, pacing_delay
import time

RESULTS_PATH = project_root / "eval" / "eval_results.jsonl"
completed = load_completed_keys(RESULTS_PATH)
print(f"{len(completed)} (strategy, question) pairs already done\n")

quota_hit = False
for strategy_name, strategy in strategies.items():
    if quota_hit:
        break
    for item in eval_set:
        key = (strategy_name, item["id"])
        if key in completed:
            continue
        try:
            contexts = call_with_backoff(lambda: strategy.retrieve(item["question"]))
            answer = call_with_backoff(lambda: strategy.run(item["question"]))
            append_jsonl(RESULTS_PATH, {
                "strategy": strategy_name, "id": item["id"],
                "question": item["question"], "question_type": item["question_type"],
                "contexts": [d.page_content for d in contexts],
                "answer": answer, "ground_truth": item.get("ground_truth", ""),
            })
            completed.add(key)
            print(f"  [{strategy_name}] {item['id']} done")
        except Exception as e:
            if is_daily_quota_error(e):
                print(f"\nDAILY QUOTA EXHAUSTED at [{strategy_name}] {item['id']}. Resume tomorrow — checkpoint saved.\n")
                quota_hit = True
                break
            print(f"  [{strategy_name}] {item['id']} FAILED: {e}")
        time.sleep(pacing_delay(strategy_name))
    if not quota_hit:
        print(f"=== Finished {strategy_name} ===\n")

374 (strategy, question) pairs already done

=== Finished naive ===

=== Finished multi_query ===

=== Finished rag_fusion ===

=== Finished hyde ===

=== Finished step_back ===

  [decomposition] b3_dist_pinn_dd_001 done
  [decomposition] b3_dist_pinn_dd_003 done
  [decomposition] b3_dist_pinn_dd_004 done
  [decomposition] b3_dist_pinn_dd_005 done
=== Finished decomposition ===

